In [6]:
import numpy as np
from pathlib import Path
import warnings

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.pipeline import make_pipeline
warnings.filterwarnings("ignore")

DATA_DIR = Path("..") / "initial_data"

In [7]:
def suggest_surrogate_point(X, y, model_type="gradientboost", resolution=0.05, random_state=0, func_id=None):
    """
    Fit a surrogate model, predict on a grid, return point with max prediction.
    """
    d = X.shape[1]
    
    # Build model
    if model_type == "gradientboost":
        # Per-function tuning for GradientBoosting
        if func_id == 1:
            # F1: Values are tiny (1e-9 to 1e-291) - log transform to see differences
            y_transformed = np.log10(np.abs(y) + 1e-300)
            lr, n_est, depth = 0.05, 150, 2
        elif func_id == 2:
            # F2: Stuck, need to find pattern - medium LR
            y_transformed = y
            lr, n_est, depth = 0.1, 100, 3
        elif func_id == 3:
            # F3: Improving, refine carefully - lower LR
            y_transformed = y
            lr, n_est, depth = 0.05, 150, 2
        else:
            y_transformed = y
            lr, n_est, depth = 0.05, 150, 2
        model = GradientBoostingRegressor(n_estimators=n_est, max_depth=depth, learning_rate=lr, random_state=random_state)
        model.fit(X, y_transformed)
    elif model_type == "knn":
        k = min(5, len(X) - 1)
        model = KNeighborsRegressor(n_neighbors=k)
        model.fit(X, y)
    elif model_type == "poly2":
        model = make_pipeline(PolynomialFeatures(degree=2, interaction_only=False, include_bias=True), LinearRegression(), memory=None)
        model.fit(X, y)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")
    
    # Grid resolution based on dimension
    if d <= 3:
        res = resolution
    elif d <= 5:
        res = 0.1
    else:
        res = 0.15
    
    # Generate grid
    grids = [np.arange(0, 1 + res, res) for _ in range(d)]
    mesh = np.meshgrid(*grids, indexing='ij')
    candidates = np.column_stack([m.ravel() for m in mesh])
    
    # Predict and find best
    predictions = model.predict(candidates)
    best_idx = np.argmax(predictions)
    best_x = candidates[best_idx]
    
    return best_x, model

In [8]:
def suggest_kmeans_cluster_point(X, y, n_clusters=3):
    """
    Use K-Means to find the best-performing cluster and return its center.
    """
    kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
    labels = kmeans.fit_predict(X)
    
    # Find cluster with highest average y
    cluster_scores = []
    for c in range(n_clusters):
        mask = labels == c
        if mask.sum() > 0:
            cluster_scores.append((c, y[mask].mean(), y[mask].max(), mask.sum()))
        else:
            cluster_scores.append((c, -np.inf, -np.inf, 0))
    
    cluster_scores.sort(key=lambda x: x[1], reverse=True)
    
    print(f"    Cluster analysis:")
    for c, avg, best, count in cluster_scores:
        print(f"      Cluster {c}: n={count}, avg={avg:.4f}, best={best:.4f}")
    
    best_cluster_id = cluster_scores[0][0]
    best_center = np.clip(kmeans.cluster_centers_[best_cluster_id], 0, 1)
    
    return best_center, best_cluster_id

In [9]:
def parse_multiline_lists(filepath):
    with open(filepath) as f:
        content = f.read()
    rounds = []
    buffer = ""
    bracket_count = 0
    for char in content:
        buffer += char
        if char == '[':
            bracket_count += 1
        elif char == ']':
            bracket_count -= 1
            if bracket_count == 0 and buffer.strip():
                rounds.append(buffer.strip())
                buffer = ""
    return rounds

# Load data - UPDATE FILE NAMES HERE
input_lines = parse_multiline_lists("inputs_m22.txt")
output_lines = parse_multiline_lists("outputs_m22.txt")

inputs_rounds = [eval(line, {"np": np, "array": np.array}) for line in input_lines]
outputs_rounds = [eval(line, {"np": np}) for line in output_lines]
n_rounds = len(inputs_rounds)
print(f"Loaded {n_rounds} rounds")

Loaded 10 rounds


In [10]:
# WEEK 11 STRATEGY
# F1: GradientBoosting - capture non-linear patterns
# F2: GradientBoosting - handle moderate non-linearity  
# F3: GradientBoosting - capture local patterns
# F4: Poly2 - smooth unimodal function
# F5: KNN - corner optimum, local neighborhood
# F6: K-Means - find best cluster, avoid bad corners
# F7: KNN - local neighborhood
# F8: Poly2 - perfect polynomial fit

surrogate_map = {
    1: "gradientboost",
    2: "gradientboost",
    3: "cluster",  # Stay in proven region
    4: "poly2",
    5: "cluster",  # Corner optimum - stay near best
    6: "kmeans",
    7: "cluster",  # Stay in proven region
    8: "cluster",  # Stay in proven region
}

print("=" * 60)
print(f"WEEK {n_rounds + 1} QUERIES")
print("=" * 60)

for i in range(1, 9):
    # Load initial data
    X = np.load(DATA_DIR / f"function_{i}" / "initial_inputs.npy")
    y = np.load(DATA_DIR / f"function_{i}" / "initial_outputs.npy")

    # Append all rounds
    for r in range(n_rounds):
        X = np.vstack([X, np.array(inputs_rounds[r][i-1]).reshape(1, -1)])
        y = np.append(y, float(outputs_rounds[r][i-1]))

    best_y = y.max()
    model_type = surrogate_map[i]
    
    print(f"\nF{i}: {model_type}")
    
    if model_type == "kmeans":
        best_x, cluster_id = suggest_kmeans_cluster_point(X, y, n_clusters=3)
        strategy = f"K-Means (cluster {cluster_id})"
    elif model_type == "cluster":
        # Cluster center: mean of top K results
        top_k = 5
        top_indices = np.argsort(y)[-top_k:]
        top_X = X[top_indices]
        cluster_center = top_X.mean(axis=0)
        
        # Detect corner optimum: if best point has dimensions AT 0 or 1 (tight threshold)
        best_point = X[np.argmax(y)]
        boundary_threshold = 0.05  # Must be very close to 0 or 1
        dims_at_boundary = np.sum((best_point < boundary_threshold) | (best_point > 1 - boundary_threshold))
        
        if dims_at_boundary >= len(best_point) * 0.75:  # At least 75% at exact boundary
            # Corner optimum detected - push toward boundaries
            corner_x = np.where(best_point > 0.5, 1.0, 0.0)
            # Check if corner is duplicate
            is_dup = np.any(np.all(np.isclose(X, corner_x, atol=1e-4), axis=1))
            if is_dup:
                # Probe slightly inside the corner
                best_x = np.where(best_point > 0.5, 0.95, 0.05)
                strategy = f"Corner probe (corner was duplicate)"
            else:
                best_x = corner_x
                strategy = f"Corner push ({dims_at_boundary}/{len(best_point)} dims at boundary)"
        else:
            # Interior optimum - use cluster center
            best_x = np.clip(cluster_center, 0, 1)
            strategy = f"Cluster center (top {top_k})"
    else:
        best_x, _ = suggest_surrogate_point(X, y, model_type=model_type, random_state=40+i, func_id=i)
        strategy = f"Surrogate ({model_type})"
    
    query_str = "-".join(f"{v:.6f}" for v in best_x)
    is_dup = np.any(np.all(np.isclose(X, best_x, atol=1e-4), axis=1))
    
    print(f"  Best so far: {best_y:.6f}")
    print(f"  Query: {query_str}")
    if is_dup:
        print("  WARNING: DUPLICATE!")

WEEK 11 QUERIES

F1: gradientboost
  Best so far: 0.000000
  Query: 0.650000-0.550000

F2: gradientboost
  Best so far: 0.611205
  Query: 0.700000-0.850000

F3: cluster
  Best so far: -0.004596
  Query: 0.433840-0.459570-0.399147

F4: poly2
  Best so far: 0.494756
  Query: 0.400000-0.400000-0.400000-0.400000

F5: cluster
  Best so far: 8662.482500
  Query: 0.950000-0.950000-0.950000-0.950000

F6: kmeans
    Cluster analysis:
      Cluster 1: n=13, avg=-0.7395, best=-0.2575
      Cluster 0: n=6, avg=-1.4131, best=-0.8292
      Cluster 2: n=11, avg=-1.6478, best=-1.2100
  Best so far: -0.257541
  Query: 0.563065-0.361635-0.679046-0.853155-0.098642

F7: cluster
  Best so far: 2.066401
  Query: 0.084586-0.417742-0.126704-0.222270-0.371484-0.707214

F8: cluster
  Best so far: 9.942989
  Query: 0.060131-0.138395-0.082094-0.075867-0.959153-0.479208-0.106114-0.676457
